# 9.1 想讓模型學到哪些行為？

# 第 9 章：誠實、安全與適當回應

前置：第7章SFT與第8章分項評估。用能驗證真假的小世界教誠實、修正前提與安全邊界，不把課堂成功推廣成任意開放世界安全保證。外部PKU資料的非商用授權與原始安全標籤要保留。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像分清「借自己的鑰匙」和「拿別人的鑰匙」**

主題相同也可有不同權限，不能只靠關鍵字拒絕。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/safety.svg")))

**先想一想：**選成較佳的回答一定安全嗎？

有用、誠實與安全是不同判讀：拒絕可以安全卻沒幫助；流暢解釋可以有用卻包含錯誤。標註先分項，再討論取捨。

**把直覺寫成數學：**資料schema保存各面向標籤，不能用一個chosen位元替代所有屬性。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
record = {"prompt": "盒中有幾顆球？", "answer": "資訊不足，請提供數量。", "helpful": True, "honest": True, "safe": True}
print(record)
pair = {"response0_safe": False, "response1_safe": False, "safer_response_id": 0}
assert not (pair["response0_safe"] or pair["response1_safe"])

**如何讀結果：**兩個回答都不安全時，較安全並不是可直接SFT的安全正例。

**只改一件事：**只改誠實標籤，說明它能否由安全標籤推論。
